# Procesamiento Final
Equipo 6

In [36]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [37]:
df = pd.read_csv('ConglomeradoTalux.csv', low_memory=False)
df.drop('Unnamed: 0', axis=1, inplace=True)

## Cliente/Leads

In [38]:
import csv
import random
from datetime import date, timedelta

inicio = date(1960, 1, 1)
fin = date(2000, 12, 31)
delta = fin - inicio

random.seed(33)
fechas = []
for i in range(1, 13935):
    dias_aleatorios = random.randint(0, delta.days)
    fecha = inicio + timedelta(days=dias_aleatorios)
    fecha_str = fecha.strftime("%d/%m/%Y")
    fechas.append(fecha_str)

In [39]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

clientes = df[['Cliente','Nombre cliente','E-mail','Telefono 1','Telefono 2','Localidad','C.P.','Colonia']].drop_duplicates(subset='Cliente').set_index('Cliente')

clientes['Nombre cliente'] = list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).nombre)
clientes['Apellido'] = list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False, header=0).apellidos)
clientes['Genero'] = list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False, header=0).genero)
clientes['Fecha_de_nacimiento'] = fechas
clientes['id_empresa'] = [(i % 1000) + 1 for i in range(clientes.shape[0])]
clientes['id_ejecutivo'] = [(i % 15) + 1 for i in range(clientes.shape[0])]

clientes.reset_index(drop=False, inplace=True)

clientes.dropna(subset=['Cliente'], inplace=True)

clientes.rename(columns={'Cliente':'id_cliente'}, inplace=True)
clientes.rename(columns={'Nombre cliente':'Nombre'}, inplace=True)

#La columna es_lead toma el valor de verdadero cuando el id del cliente no aparece en la tabla ventas.
clientes['Es_lead'] = np.random.choice([True, False], size=len(clientes), p = [0.05,0.95])

clientes['Lead_Cualificacion'] = np.nan
clientes['Satisfaccion'] = np.nan

In [40]:
# Simulamos datos que sean nulos y que no permitan nulls

clientes.loc[clientes['E-mail'].isna(), 'E-mail'] = (
    clientes.loc[clientes['E-mail'].isna(), 'Nombre'].str[:4].str.lower() + '_' +
    clientes.loc[clientes['E-mail'].isna(), 'Apellido'].str[:4].str.lower() + '@gmail.com'
)

In [41]:
# Reordenamos columnas para que siga el modelo de datos

clientes = clientes[['id_cliente', 'Nombre', 'Apellido', 'Genero', 'Fecha_de_nacimiento', 'E-mail', 'Telefono 1', 'Telefono 2', 'id_empresa', 'id_ejecutivo', 'Localidad', 'C.P.', 'Colonia', 'Es_lead']]

#Quitamos el único valor erróneo identificado
clientes = clientes[clientes['id_cliente'] != 'BI']

## Ventas

In [42]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

ventas = df[['T.factura','Cliente','VIN']].drop_duplicates(subset='T.factura').set_index('T.factura')
ventas.reset_index(drop=False, inplace=True)
ventas.dropna(inplace=True)

ventas['id_ejecutivo'] = [(i % 15) + 1 for i in range(ventas.shape[0])]

dates = pd.date_range(start='2017-01-01', end=pd.Timestamp.today(), freq='D')
sample = np.random.choice(dates.strftime('%d/%m/%Y'), size=1313, replace=True)
ventas['Fecha'] = sample

ventas['Duracion contrato'] = np.random.choice([24,36,48], size=1313, replace=True)
ventas['Contrato finalizado'] = pd.to_datetime(ventas['Fecha'], dayfirst = True) < datetime.today() - timedelta(days=4*365)

ventas.reset_index(drop=False, inplace=True)
ventas['index'] = ventas['index'].astype(int) + 1

In [43]:
ventas.rename(columns={'index': 'id_venta', 'T.factura': 'Monto', 'Cliente': 'id_cliente'}, inplace=True)

# Reordenamos columnas para que siga el modelo de datos

ventas = ventas[['id_venta', 'id_cliente', 'VIN', 'id_ejecutivo', 'Fecha', 'Monto',
                 'Duracion contrato', 'Contrato finalizado']]

ventas.loc[ventas['id_cliente'].isin(clientes[clientes['Es_lead'] == True]['id_cliente']), 'Monto'] = np.nan
ventas.loc[ventas['id_cliente'].isin(clientes[clientes['Es_lead'] == True]['id_cliente']), 'Contrato finalizado'] = False

ventas['Monto'] = np.abs(ventas['Monto'])

## Autos

In [44]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

autos = df[['VIN','Modelo','Color','Año Vehiculo']].drop_duplicates(subset='VIN').set_index('VIN')
autos.reset_index(drop=False, inplace=True)
autos = autos[autos['VIN'].isin(ventas['VIN'].unique())]

autos['Marca'] = list(pd.read_csv('Simulados/carros.csv', low_memory=False).Manufacturer)
autos['Año Vehiculo'] = list(pd.read_csv('Simulados/carros.csv', low_memory=False).Model_Year)
autos

,VIN,Modelo,Color,Año Vehiculo,Marca
0,1HGCY1673SA900003,ACCORD PRIME,NEGRO CRISTA,2025,Honda (USA)
2,1HGCY1677SA900067,ACCORD PRIME,BLANCO PLATI,2025,Honda (USA)
5,1HGCV1619LA900142,ACCORD EX CVT,BLANCO PLATI,2020,Honda (USA)
6,1HGCV1615LA900090,ACCORD EX CVT,PLATA LUNAR,2020,Honda (USA)
7,1HGCV1610LA900370,ACCORD EX CVT,PLATA LUNAR,2020,Honda (USA)
...,...,...,...,...,...
30372,5KBYF4885EB801813,Pilot Touring,PLATA DIAMAN,2014,Honda (Thailand)
30406,5KBYF6895GB801169,Pilot Touring,AZUL OBSIDIA,2016,Honda (Thailand)
30416,5KBYF689XHB801122,Pilot Touring,PLATA LUNAR,2017,Honda (Thailand)
30461,5FPYK1654BB801368,Ridgeline Rtl 2011,BLANCO MARFI,2011,Honda (USA)


## Ejecutivos

In [45]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

ejecutivos = pd.DataFrame(
    {'id_ejecutivo' : [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
     'Nombre': list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).nombre)[300:315],
     'Apellido': list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).apellidos)[600:615],
     'Genero': list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).genero)[300:315],
     'Fecha_de_nacimiento' : fechas[600:615],
     'E-mail' : list(clientes['E-mail'][300:315]),
     'Telefono' : ['9191175013','9191094789','9191088722','9632587412','9615897548','9191875013','9191594789','9198788722','9632596412','9615637548','9191175363','9541094789','9871088722','9632512412','9615007548'],
     'Localidad' : list(clientes['Localidad'][900:915]),
     'C.P.' : clientes['C.P.'].unique()[-15:],
     'Colonia' : clientes['Colonia'].unique()[-15:],
     'Fecha_de_ingreso': ["01/01/2019", "01/07/2019", "02/01/2020", "08/07/2020", "04/09/2021", "01/07/2021", "11/11/2022", "12/11/2022", "04/04/2023", "09/09/2023", "30/05/2024", "03/09/2024", "05/11/2025", "07/12/2025", "11/10/2026"]})

## Empresa

In [46]:
empresas = pd.DataFrame({
    "id_empresa" : list(range(1,1001)),
    "Nombre" : list(pd.read_csv('Simulados/Empresas.csv', low_memory=False).Empresa),
    "Giro" : np.random.choice(['Manufactura', 'Servicios', 'Comercial'], p=[1/3,1/3,1/3], size = 1000),
    "Tamaño personal" : np.random.randint(10,3000, size = 1000),
    "Localidad" : list(clientes['Localidad'][10000:11000]),
    "C.P." : list(clientes['C.P.'][10000:11000]),
    "Colonia" : list(clientes['Colonia'][10000:11000]),
})

## Arreglando Problemitas Finales

In [47]:
clientes['Fecha_de_nacimiento'] = pd.to_datetime(clientes['Fecha_de_nacimiento'], format = "%d/%m/%Y")
clientes['Telefono 1'] = pd.to_numeric(clientes['Telefono 1'], errors="coerce").astype('float64').astype('Int64')
clientes['Telefono 2'] = pd.to_numeric(clientes['Telefono 2'], errors="coerce")
clientes['Telefono 2'] = clientes['Telefono 2'].apply(lambda x: int(float(x)) if pd.notna(x) else None).astype('Int64')
clientes['C.P.'] = pd.to_numeric(clientes['C.P.'], errors="coerce").astype('float64').astype('Int64')

ventas['Fecha'] = pd.to_datetime(ventas['Fecha'], format = "%d/%m/%Y")

autos['Año Vehiculo'] = pd.to_numeric(autos['Año Vehiculo'], errors="coerce").astype('float64').astype('Int64')

ejecutivos['Telefono'] = pd.to_numeric(ejecutivos['Telefono'], errors="coerce").astype('float64').astype('Int64')
ejecutivos['C.P.'] = pd.to_numeric(ejecutivos['C.P.'], errors="coerce").astype('float64').astype('Int64')
ejecutivos['Fecha_de_nacimiento'] = pd.to_datetime(ejecutivos['Fecha_de_nacimiento'], format = "%d/%m/%Y")
ejecutivos['Fecha_de_ingreso'] = pd.to_datetime(ejecutivos['Fecha_de_ingreso'], format = "%d/%m/%Y")

empresas['C.P.'] = pd.to_numeric(empresas['C.P.'], errors="coerce").astype('float64').astype('Int64')

## Exportar Tablas

In [48]:
clientes.to_csv('TablasFinales/clientes_leads.csv', index=False)
autos.to_csv('TablasFinales/autos.csv', index=False)
ejecutivos.to_csv('TablasFinales/ejecutivos.csv', index=False)
ventas.to_csv('TablasFinales/ventas.csv', index=False)
empresas.to_csv('TablasFinales/empresas.csv', index=False)